In [1]:
import json
from helper import helpers as hp
import pandas as pd
from tqdm import tqdm

In [2]:
import importlib

# Data Loading

In [3]:
kpi_concept_meta_root = "./data/concept/"
respondent_level_meta_root = "./data/respondent/"
local_data_root = "./data/"

In [9]:
# get the concept file
# cid_name = "cid_concept_us_food.json"
cid_name = "cid_concept_us_personalcare.json"
date = "0617"


with open(kpi_concept_meta_root + cid_name, encoding="utf-8") as f:
    cid_concept = json.load(f)

In [5]:
for k, v in cid_concept.items():
    cid_concept[k] = v['content']

# rephrasing

In [6]:
import importlib
from augmentation import rephrasing
importlib.reload(rephrasing)


from augmentation.rephrasing import ConceptRephraser, RephraseConfig, Tone, PointOfView, ContentOrder

# Initialize the rephraser with the API key from the module
rephraser = ConceptRephraser(model="gpt-4o", temperature=0.5)

In [7]:
import random

def multi_rephrase_concept(concept_text, n_variations=8):
    """
    Generate n random variations of a concept using different rephrasing configurations.
    
    Args:
        concept_text: The original concept text
        n_variations: Number of variations to generate (default: 5)
    
    Returns:
        List of dicts with 'config' and 'text' keys
    """
    tones = [None, Tone.CLINICAL, Tone.MARKETING, Tone.CONVERSATIONAL]
    povs = [None, PointOfView.SECOND_PERSON, PointOfView.THIRD_PERSON]
    orders = [None, ContentOrder.PROBLEM_FIRST, ContentOrder.FEATURE_FIRST, ContentOrder.BENEFIT_FIRST]
    length_options = [True, False]
    
    # Generate unique random configurations
    seen_configs = set()
    variations = []
    
    while len(variations) < n_variations:
        config_tuple = (
            random.choice(tones),
            random.choice(povs),
            random.choice(orders),
            random.choice(length_options)
        )
        
        # Skip if we've already used this exact config
        if config_tuple in seen_configs:
            continue
        seen_configs.add(config_tuple)
        
        tone, pov, order, change_len = config_tuple
        
        config = RephraseConfig(
            change_length=change_len,
            tone=tone,
            point_of_view=pov,
            content_order=order,
        )
        
        result = rephraser.rephrase(concept_text, config)
        
        variations.append({
            'config': {
                'tone': tone.value if tone else None,
                'point_of_view': pov.value if pov else None,
                'content_order': order.value if order else None,
                'change_length': change_len,
            },
            'text': result.rephrased_text,
            'word_count': result.new_word_count,
        })
    
    return variations


# Test it
sample_cid = list(cid_concept.keys())[0]
sample_text = cid_concept[sample_cid]
print(f"Original ({len(sample_text.split())} words):\n{sample_text}\n")
print("=" * 60)

variations = multi_rephrase_concept(sample_text, n_variations=8)

for i, var in enumerate(variations, 1):
    print(f"\n[Variation {i}] Config: {var['config']}")
    print(f"Word count: {var['word_count']}")
    print(var['text'])
    print("-" * 60)

Original (132 words):
Clinique Acne Solutions™ is a dermatologist-tested skincare and makeup line that treats acne and helps maintain clear skin long term. Clinique Acne Solutions™ Drying Lotion is a fast-acting liquid spot treatment designed to target occasional breakouts overnight (8 hours). Formulated with: 2% Salicylic Acid: Promotes the gentle removal of dead, dulling surface skin cells which can clog pores and cause breakouts. Rapid-Dry Up Technology: An alcohol-based solution of niacinamide, zinc PCA, zinc oxide, barium sulfate, titanium dioxide, and kaolin works quickly to dry up blemishes and reduce their appearance. Glycerin: Helps counter dryness and soothe irritated skin. Use a Q-tip to apply to targeted areas. Formula dries blue and should be left on overnight for best results. Safe for sensitive skin. Safe for acne-prone skin. Dermatologist Tested. Allergy Tested. 100% Fragrance Free.


[Variation 1] Config: {'tone': None, 'point_of_view': 'second_person', 'content_order'

In [8]:
from tqdm import tqdm

variation_cid_concept = {}

for k, v in tqdm(cid_concept.items(), desc="Processing concepts"):
    variations = multi_rephrase_concept(v, n_variations=8)
    sub_variations = {i+1: var for i, var in enumerate(variations)}
    variation_cid_concept[k] = sub_variations

Processing concepts:   0%|          | 0/12 [00:00<?, ?it/s]

Processing concepts: 100%|██████████| 12/12 [04:43<00:00, 23.60s/it]


In [23]:
# file_name = "0414_variation_cid_concept.json"
file_name = date + "_variation_" + cid_name
with open(local_data_root + file_name, 'w', encoding="utf-8") as f:
    json.dump(variation_cid_concept, f, indent=2, ensure_ascii=False)

In [22]:
variation_cid_concept

{'1': {1: {'config': {'tone': 'marketing',
    'point_of_view': 'third_person',
    'content_order': 'benefit_first',
    'change_length': True},
   'text': 'Experience clearer skin with Clinique Acne Solutions™! Users will love the fast-acting Drying Lotion, a game-changer for overnight breakout treatment in just 8 hours. With 2% Salicylic Acid, it gently exfoliates dead skin cells, preventing clogged pores. The Rapid-Dry Up Technology, featuring niacinamide, zinc PCA, and kaolin, swiftly dries blemishes, reducing their appearance. Glycerin ensures skin stays hydrated and soothed. Simply apply with a Q-tip, let it dry blue, and leave overnight for optimal results. Perfect for sensitive and acne-prone skin. Dermatologist and Allergy Tested. 100% Fragrance Free.',
   'word_count': 88},
  2: {'config': {'tone': 'conversational',
    'point_of_view': 'second_person',
    'content_order': 'benefit_first',
    'change_length': True},
   'text': "You want clear skin, right? Clinique Acne Sol

In [15]:
result = {}
for k, v in variation_cid_concept.items():
    val = 0
    for i, var in v.items():
        if var['config']['change_length']:
            val += 1
    result[k] = val

    
        